# 📊 Automated Trading Analysis & Optimization System

Nhập mã chứng khoán VN (`FPT`, `HPG`) hoặc cặp Crypto Binance (`BTCUSDT`, `ETHUSDT`) → hệ thống tự chạy:

**Data → Cleaning → TA + FA → Walk-Forward Optimization → Khuyến nghị MUA/BÁN/ĐỨNG NGOÀI**

- SL dựa trên ATR (không dùng % cố định), TP theo R-multiple
- Win Rate đếm thật từ backtest out-of-sample, không ước lượng

In [ ]:
from trading_system.main import analyze
from trading_system.decision import render_markdown, render_json
from IPython.display import Markdown

SYMBOL = "FPT"        # 👈 đổi mã ở đây: FPT, HPG, BTCUSDT, ETHUSDT...
LOOKBACK_YEARS = 5

In [ ]:
decision = analyze(SYMBOL, lookback_years=LOOKBACK_YEARS)
Markdown(render_markdown(decision))

## 🔍 Đào sâu: equity curve & chi tiết từng lệnh của bộ tham số tối ưu

In [ ]:
import pandas as pd
from trading_system.data import route_asset, fetch_vn_ohlcv, fetch_crypto_ohlcv, clean_ohlcv
from trading_system.indicators import compute_ta_features
from trading_system.backtester import run_backtest
from trading_system.config import AssetType, get_cost_model, get_market_constraints

asset = route_asset(SYMBOL)
raw = fetch_vn_ohlcv(SYMBOL, LOOKBACK_YEARS) if asset == AssetType.VN_STOCK else fetch_crypto_ohlcv(SYMBOL, "1d", LOOKBACK_YEARS)
df, _ = clean_ohlcv(raw)
feats = compute_ta_features(df)

bt = run_backtest(df, feats, decision.best_params, get_cost_model(asset), get_market_constraints(asset), keep_trades=True)
print(f"Toàn bộ lịch sử: {bt.n_trades} lệnh | win {bt.win_rate:.0%} | PF {bt.profit_factor:.2f} | maxDD {bt.max_drawdown:.0%}")
bt.trades[["entry_date", "exit_date", "return", "r_multiple", "bars_held", "reason"]].tail(15)

In [ ]:
# Equity curve của chiến lược vs Buy & Hold (matplotlib có sẵn theo pandas)
import numpy as np
import matplotlib.pyplot as plt

eq = (1 + pd.Series([t for t in bt.trades["return"]], index=bt.trades["exit_date"])).cumprod()
bh = df["close"] / df["close"].iloc[0]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(bh.index, bh.values, label="Buy & Hold", alpha=0.6)
ax.step(eq.index, eq.values, label=f"Strategy ({decision.best_params['rsi_period']}-RSI, {decision.best_params['atr_sl_mult']}xATR SL)", where="post")
ax.set_title(f"{SYMBOL} — Strategy vs Buy & Hold")
ax.legend(); ax.grid(alpha=0.3)
plt.show()

## 📤 Xuất JSON (cho hệ thống downstream / API)

In [ ]:
print(render_json(decision))